# AI应用基础 · 工单M3.2《产线实时缺陷检测系统》 — 教师参考版
第9周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m3yolo"], check=True)
print("数据已生成: data/")

## 参考实现 `m32`（含依赖导入）

In [ ]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m32():
    import cv2
    try:
        import onnxruntime as ort
    except ImportError:
        print("本机未装onnxruntime: 参考run_ultralytics方案或装onnxruntime")
        return
    sess = ort.InferenceSession("data/weights/yolov8n.onnx",
                                providers=["CPUExecutionProvider"])
    import glob
    total = {}
    for p in sorted(glob.glob("data/yolo_dataset/images/*.npy"))[:10]:
        img = np.load(p).astype(np.uint8)
        x = cv2.resize(img, (640, 640)).astype(np.float32) / 255.0
        out = sess.run(None, {"images": x.transpose(2, 0, 1)[None]})[0][0]
        cls = out[4:]                       # (80,8400)
        n = int((cls.max(0) > 0.4).sum())
        total[p] = n
        print(p.split("/")[-1], "conf>0.4 检出:", n)
    print("阈值实验: 降低conf检出↑误检↑; 产线建议conf=0.4~0.6,"
          "imgsz=320在CPU上平衡精度与FPS。")


## 执行与验证

In [ ]:
m32()